# Notebook 04 — End-to-End Pipeline Demo

Companion to `04-building-the-sequential-pipeline.md`. This chains together simplified/mocked
versions of all three stages into a single `run_pipeline()` function, run on one synthetic
example, and prints the final structured citation-detection output — the same
OCR -> YOLO(+NMS) -> CNN flow described throughout this course, in miniature.

- **Stage 1 (OCR)**: `fake_ocr()` — a mocked stand-in for Notebook 01 / Chapter 01, returning
  synthetic word boxes with line/baseline metadata instead of calling a real OCR engine.
- **Stage 2 (YOLO + NMS)**: `yolo_stage_propose()` — reuses the same IOU/NMS logic built from
  scratch in Notebook 02, applied here to dict-based candidates so text/metadata can travel
  alongside each box through the pipeline.
- **Stage 3 (CNN classifier)**: `cnn_stage_classify()` — a lightweight **stub** standing in
  for the trained CNN from Notebook 03. In production this call would run the real trained
  model on the cropped candidate's pixels; here it approximates the same true/false-positive
  decision using the same kind of geometric signal (elevation above the line baseline) the
  real CNN learns implicitly from crops, so the pipeline's control flow can be demonstrated
  end to end without needing to load a saved model file.

Pure Python + numpy — no external dependencies beyond what's in the standard toolchain.


## Stage 1 (mocked): OCR

Returns a synthetic OCR result for one document page: a sentence containing one **true**
citation superscript ("1", small + elevated after "40%"), plus a **decoy** — a page number
("7") elsewhere on the page that is also small, but sits *on* its own line's baseline rather
than above it. The decoy exists specifically to demonstrate Stage 3 correctly rejecting a
small-but-not-elevated false positive that Stage 2 will still propose as a candidate.


In [1]:
def fake_ocr(image_id="doc_page_007"):
    """Mocked Stage 1 (OCR). Returns words with bounding boxes plus line baseline/font
    metadata, mirroring the structured OCR output described in Chapter 01 and produced (for
    real) in Notebook 01."""
    sentence_baseline_y, sentence_font_px = 216, 16
    footer_baseline_y, footer_font_px = 392, 16

    words = [
        {"text": "reduced", "bbox": (40, 200, 96, 216),  "line_baseline_y": sentence_baseline_y, "line_font_px": sentence_font_px},
        {"text": "onset",   "bbox": (100, 200, 148, 216), "line_baseline_y": sentence_baseline_y, "line_font_px": sentence_font_px},
        {"text": "by",      "bbox": (152, 200, 168, 216), "line_baseline_y": sentence_baseline_y, "line_font_px": sentence_font_px},
        {"text": "40%",     "bbox": (172, 200, 204, 216), "line_baseline_y": sentence_baseline_y, "line_font_px": sentence_font_px},
        {"text": "1",       "bbox": (206, 190, 212, 200), "line_baseline_y": sentence_baseline_y, "line_font_px": sentence_font_px},  # true superscript citation marker
        {"text": "in",      "bbox": (216, 200, 232, 216), "line_baseline_y": sentence_baseline_y, "line_font_px": sentence_font_px},
        {"text": "trial",   "bbox": (236, 200, 276, 216), "line_baseline_y": sentence_baseline_y, "line_font_px": sentence_font_px},
        {"text": "7",       "bbox": (400, 380, 408, 392), "line_baseline_y": footer_baseline_y,   "line_font_px": footer_font_px},    # decoy: page number, small but NOT elevated
    ]
    return {"image_id": image_id, "words": words}


ocr_result = fake_ocr()
print(f"OCR found {len(ocr_result['words'])} words on {ocr_result['image_id']}:\n")
for w in ocr_result["words"]:
    print(w["text"], w["bbox"])


OCR found 8 words on doc_page_007:

reduced (40, 200, 96, 216)
onset (100, 200, 148, 216)
by (152, 200, 168, 216)
40% (172, 200, 204, 216)
1 (206, 190, 212, 200)
in (216, 200, 232, 216)
trial (236, 200, 276, 216)
7 (400, 380, 408, 392)


## Stage 2 (mocked): YOLO candidate proposal + NMS

In production, a trained YOLO model scans the image directly. Here we approximate its
*behavior* — proposing (possibly duplicated/jittered) candidate boxes for any glyph that is
small relative to its line's font size — and then clean duplicates up with the same IOU/NMS
logic built from scratch in `02_yolo_object_detection_demo.ipynb`. Note this stage
deliberately assigns a **high confidence to any small digit-like glyph regardless of
elevation** (mirroring Chapter 02/03's point that YOLO over-triggers on small,
citation-marker-*shaped* things like page numbers) — the elevation check that actually
separates the true marker from the decoy is left entirely to Stage 3.


In [2]:
def iou(bbox_a, bbox_b):
    xa1, ya1, xa2, ya2 = bbox_a
    xb1, yb1, xb2, yb2 = bbox_b
    inter_x1, inter_y1 = max(xa1, xb1), max(ya1, yb1)
    inter_x2, inter_y2 = min(xa2, xb2), min(ya2, yb2)
    inter_w, inter_h = max(0, inter_x2 - inter_x1), max(0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h
    area_a = (xa2 - xa1) * (ya2 - ya1)
    area_b = (xb2 - xb1) * (yb2 - yb1)
    union = area_a + area_b - inter_area
    return inter_area / union if union > 0 else 0.0


def non_max_suppression(candidates, iou_threshold=0.4):
    """Same algorithm as Notebook 02, applied to a list of dicts (each with a 'bbox' and
    'score' key) instead of a raw numpy array, so the source word's text/metadata can travel
    through the pipeline alongside each box."""
    ordered = sorted(candidates, key=lambda c: c["score"], reverse=True)
    keep = []
    while ordered:
        current = ordered.pop(0)
        keep.append(current)
        ordered = [c for c in ordered if iou(current["bbox"], c["bbox"]) <= iou_threshold]
    return keep


def yolo_stage_propose(ocr_result, size_ratio_threshold=0.75):
    """Mocked Stage 2. Proposes candidate boxes for any word whose bounding-box height is
    small relative to its line's font size (a plausible superscript *shape*), including a
    couple of jittered/duplicate raw detections per true candidate -- exactly what a
    single-shot detector emits before NMS -- then cleans duplicates up with NMS."""
    raw = []
    for w in ocr_result["words"]:
        x1, y1, x2, y2 = w["bbox"]
        h = y2 - y1
        if h <= size_ratio_threshold * w["line_font_px"]:
            base_conf = 0.88 if w["text"].isdigit() else 0.35
            for dx, dy, conf in [(0, 0, base_conf), (1, 1, base_conf - 0.20)]:
                raw.append({
                    "text": w["text"],
                    "bbox": (x1 + dx, y1 + dy, x2 + dx, y2 + dy),
                    "score": conf,
                    "source_word": w,
                })
    return non_max_suppression(raw, iou_threshold=0.4)


candidates = yolo_stage_propose(ocr_result)
print(f"{len(candidates)} candidate region(s) survive Stage 2 (post-NMS):\n")
for c in candidates:
    print(f'{c["text"]!r:>5}  bbox={c["bbox"]}  score={c["score"]:.2f}')


2 candidate region(s) survive Stage 2 (post-NMS):

  '1'  bbox=(206, 190, 212, 200)  score=0.88
  '7'  bbox=(400, 380, 408, 392)  score=0.88


## Stage 3 (stubbed): CNN classification

A real trained CNN (Notebook 03) makes this decision from the candidate's cropped pixels. As
a stand-in that keeps this notebook self-contained and dependency-free, this stub uses the
candidate's elevation above its line's baseline as a proxy for the same visual signal the
real network learns implicitly. This is where the true citation marker (elevated ~16px above
its baseline) gets correctly separated from the page-number decoy (0px elevation — it sits
*on* its own baseline), even though both looked like strong candidates to Stage 2.


In [3]:
def cnn_stage_classify(candidate):
    """Mocked Stage 3. Approximates the trained CNN's true-superscript-vs-false-positive
    decision using the candidate's elevation above its source line's baseline -- the same
    kind of signal a real CNN picks up implicitly from the crop's pixels, per Chapter 03.
    """
    x1, y1, x2, y2 = candidate["bbox"]
    baseline_y = candidate["source_word"]["line_baseline_y"]
    elevation = baseline_y - y2  # how far above the baseline the glyph's bottom sits

    is_true_superscript = elevation > 8 and candidate["text"].isdigit()
    confidence = 0.93 if is_true_superscript else 0.81
    return {"is_superscript": is_true_superscript, "confidence": confidence, "elevation_px": elevation}


for c in candidates:
    verdict = cnn_stage_classify(c)
    print(f'{c["text"]!r:>5}  elevation={verdict["elevation_px"]:>3}px  '
          f'-> is_superscript={verdict["is_superscript"]}  (conf={verdict["confidence"]:.2f})')


  '1'  elevation= 16px  -> is_superscript=True  (conf=0.93)
  '7'  elevation=  0px  -> is_superscript=False  (conf=0.81)


## Full pipeline: chaining all three stages

Combines Stages 1-3 into one function and produces the **structured citation output** —
claim text, marker, bounding box, and confidence — that the résumé bullet's "finally identify
superscript characters" step is describing.


In [4]:
def run_pipeline(image_id="doc_page_007"):
    """End-to-end: OCR -> YOLO detect + NMS -> CNN classify -> structured citation output."""
    ocr_result = fake_ocr(image_id)
    stage2_candidates = yolo_stage_propose(ocr_result)

    citations = []
    for cand in stage2_candidates:
        verdict = cnn_stage_classify(cand)
        if not verdict["is_superscript"]:
            continue  # Stage 3 rejected this candidate (e.g. the page-number decoy)

        # Reconstruct the claim text: words on the same line, preceding this marker
        line_y = cand["source_word"]["line_baseline_y"]
        claim_words = [
            w["text"] for w in ocr_result["words"]
            if w["line_baseline_y"] == line_y and w["bbox"][2] <= cand["bbox"][0]
        ]
        citations.append({
            "image_id": image_id,
            "claim_text": " ".join(claim_words),
            "marker": cand["text"],
            "bbox": cand["bbox"],
            "confidence": verdict["confidence"],
        })
    return citations


import json

result = run_pipeline()
print(f"Structured citation-detection output ({len(result)} confirmed citation(s)):\n")
print(json.dumps(result, indent=2))


Structured citation-detection output (1 confirmed citation(s)):

[
  {
    "image_id": "doc_page_007",
    "claim_text": "reduced onset by 40%",
    "marker": "1",
    "bbox": [
      206,
      190,
      212,
      200
    ],
    "confidence": 0.93
  }
]


In [5]:
# Sanity checks: the true "1" superscript is found and linked to the right claim text;
# the page-number decoy "7" is correctly excluded despite Stage 2 flagging it as a candidate.
assert len(result) == 1, "expected exactly one confirmed citation"
assert result[0]["marker"] == "1"
assert result[0]["claim_text"] == "reduced onset by 40%"
print("OK -- pipeline correctly confirmed the true citation and rejected the page-number decoy.")


OK -- pipeline correctly confirmed the true citation and rejected the page-number decoy.


## Takeaway

This is the whole résumé bullet in miniature: OCR supplies text + geometry, a YOLO-style
detector (cleaned up with NMS) proposes small/elevated candidates without worrying about
precision, and a CNN-style classifier makes the final call per candidate — correctly telling
a true citation marker apart from a geometrically-similar decoy (here, a page number) that
Stage 2 alone could not distinguish. Swap `fake_ocr()` for a real OCR call (Notebook 01),
`yolo_stage_propose()` for a real trained detector (Notebook 02's IOU/NMS logic plus a
trained model), and `cnn_stage_classify()` for the trained model from Notebook 03, and this
`run_pipeline()` function is structurally the production system described in
`04-building-the-sequential-pipeline.md` — the same sequential architecture that took
citation tracking accuracy from 5% to 85%.
